# CNN Results — Cross-Platform Analysis

**FILE:** `notebooks/cnn_results.ipynb`
**AUTHOR:** Kananelo Chabeli — University of Cape Town

Answers the central research question:

> **Do model rankings transfer across edge hardware platforms?**

Structured in three parts:

| Part | Question |
|---|---|
| **A. Per-platform** | How does each model behave on one platform? Accuracy, latency, throughput, and the accuracy/latency frontier. |
| **B. Cross-platform** | Does the ordering of models change between platforms, and where? |
| **C. Deployability** | Which architectures compile and run at all, and what does that cost? |

Designed to grow. Register a platform in `PLATFORMS` and every figure extends automatically; the
notebook runs with a single platform and is useful from the first baseline onwards.

In [ ]:
import json, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=FutureWarning)

TASK = "classification"
ROOT = Path.cwd().parent
BENCH = ROOT / "results" / "benchmark" / TASK

# ---------------------------------------------------------------------------------------
# PLATFORM REGISTRY — the only cell to edit as hardware results arrive.
#   key      : label used in every table and figure
#   file     : CSV under results/benchmark/<task>/
#   precision: numeric precision actually executed, recorded because it is NOT uniform.
#              TensorRT INT8 keeps some layers in higher precision, whereas the Edge TPU and
#              Hailo are integer-only; comparing them without stating this would be misleading.
#   kind     : "reference" platforms are development hardware and are excluded from the
#              cross-platform ranking analysis by default.
# ---------------------------------------------------------------------------------------
PLATFORMS = {
    "GPU (FP32)":      dict(file="baseline_fp32.csv",   precision="fp32",       kind="reference"),
    # "Jetson (INT8)": dict(file="jetson_int8.csv",     precision="int8-mixed", kind="edge"),
    # "Jetson (FP16)": dict(file="jetson_fp16.csv",     precision="fp16",       kind="edge"),
    # "Coral TPU":     dict(file="tpu_int8.csv",        precision="int8",       kind="edge"),
    # "Hailo-8L":      dict(file="hailo_int8.csv",      precision="int8",       kind="edge"),
    # "Pi 5 CPU":      dict(file="rpi_cpu_int8.csv",    precision="int8",       kind="edge"),
}

# Metric name -> (column, higher_is_better, axis label)
METRICS = {
    "accuracy":   ("top1",             True,  "top-1 (%)"),
    "latency":    ("lat_p50_ms",       False, "median latency (ms)"),
    "tail":       ("lat_p99_ms",       False, "p99 latency (ms)"),
    "throughput": ("throughput_img_s", True,  "throughput (img/s)"),
}

FIGDIR = ROOT / "results" / "figures" / TASK / "results"
FIGDIR.mkdir(parents=True, exist_ok=True)
SAVE_FIGURES = True

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "legend.frameon": False,
})

def save(fig, name):
    """Write a figure to the results figure directory.

    Args:
        fig: Matplotlib figure.
        name: File stem.
    """
    if SAVE_FIGURES:
        fig.savefig(FIGDIR / f"{name}.png")
        fig.savefig(FIGDIR / f"{name}.pdf")

## 0. Load

In [ ]:
frames = []
for label, spec in PLATFORMS.items():
    path = BENCH / spec["file"]
    if not path.exists():
        print(f"  skip {label:16s} (no {spec['file']})")
        continue
    d = pd.read_csv(path)
    if "status" in d.columns:
        failed = d[d.status != "completed"]
        d = d[d.status == "completed"]
        if len(failed):
            # A model that failed to compile or run is a deployability result, not missing
            # data; it is carried through to Part C rather than dropped silently.
            print(f"  {label}: {len(failed)} did not complete -> {list(failed.model)}")
    d = d.assign(platform=label, precision=spec["precision"], kind=spec["kind"])
    frames.append(d)
    print(f"  load {label:16s} {len(d):3d} models  [{spec['precision']}, {spec['kind']}]")

if not frames:
    raise SystemExit("no platform results found — check PLATFORMS and the benchmark directory")

results = pd.concat(frames, ignore_index=True)
PLATFORM_ORDER = [p for p in PLATFORMS if p in set(results.platform)]
EDGE = [p for p in PLATFORM_ORDER if PLATFORMS[p]["kind"] == "edge"]

# Reuse the palette from the training notebook so a model keeps one colour throughout.
pal_file = ROOT / "results" / "figures" / "model_palette.json"
if pal_file.exists():
    PALETTE = {m: tuple(c) for m, c in json.loads(pal_file.read_text()).items()}
else:
    PALETTE = {}
for i, m in enumerate(sorted(results.model.unique())):
    PALETTE.setdefault(m, cm.tab20(i % 20))

print(f"\n{len(results)} rows | {results.model.nunique()} models | "
      f"{len(PLATFORM_ORDER)} platforms")
print(f"edge platforms for the ranking analysis: {EDGE or '(none yet)'}")

In [ ]:
def pivot(metric):
    """Build a model-by-platform table for one metric.

    Args:
        metric: Key of METRICS.

    Returns:
        DataFrame indexed by model with one column per platform, or None when the underlying
        column is absent everywhere.
    """
    col, _, _ = METRICS[metric]
    if col not in results.columns:
        return None
    t = results.pivot_table(index="model", columns="platform", values=col, aggfunc="first")
    cols = [p for p in PLATFORM_ORDER if p in t.columns]
    return t[cols] if cols else None


def ranked(metric, platforms=None, dropna=True):
    """Rank models on one metric, per platform.

    Rank 1 is always the best model, whichever direction the metric improves in.

    Args:
        metric: Key of METRICS.
        platforms: Restrict to these platforms.
        dropna: Keep only models measured on every selected platform, so ranks are comparable.

    Returns:
        DataFrame of ranks indexed by model, or None when unavailable.
    """
    t = pivot(metric)
    if t is None:
        return None
    if platforms:
        cols = [p for p in platforms if p in t.columns]
        if not cols:
            return None
        t = t[cols]
    if dropna:
        t = t.dropna()
    _, higher_better, _ = METRICS[metric]
    return t.rank(ascending=not higher_better, method="min")


def pareto_front(df, x, y, x_lower_better=True):
    """Select the non-dominated points of a two-objective trade-off.

    Args:
        df: Source frame.
        x: Cost column.
        y: Quality column, higher-is-better.
        x_lower_better: Whether a smaller x is preferable.

    Returns:
        Non-dominated rows sorted by x.
    """
    d = df.dropna(subset=[x, y]).sort_values(x, ascending=x_lower_better)
    keep, best = [], -np.inf
    for _, r in d.iterrows():
        if r[y] > best:
            keep.append(r); best = r[y]
    return pd.DataFrame(keep)


def heat(ax, matrix, cmap, vmin=None, vmax=None, fmt="{:.2f}", cbar_label=""):
    """Draw an annotated heatmap from a DataFrame.

    Args:
        ax: Target axes.
        matrix: DataFrame to render.
        cmap: Matplotlib colormap.
        vmin: Colour-scale minimum.
        vmax: Colour-scale maximum.
        fmt: Cell annotation format.
        cbar_label: Colour-bar label.
    """
    im = ax.imshow(matrix.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(matrix.shape[1])); ax.set_xticklabels(matrix.columns, rotation=30,
                                                              ha="right", fontsize=8)
    ax.set_yticks(range(matrix.shape[0])); ax.set_yticklabels(matrix.index, fontsize=8)
    ax.grid(False)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            v = matrix.values[i, j]
            if pd.notna(v):
                # Mid-scale cells take dark text, extremes take light, so annotations stay
                # legible against any background in the colormap.
                rel = 0.5 if vmin is None or vmax in (None, vmin) else (v - vmin) / (vmax - vmin)
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=7,
                        color="white" if rel > 0.75 or rel < 0.25 else "black")
    cb = ax.figure.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    if cbar_label:
        cb.set_label(cbar_label, fontsize=8)

---
# Part A — Per-platform behaviour

## A1. Metric tables

In [ ]:
for metric in METRICS:
    t = pivot(metric)
    if t is None or t.empty:
        continue
    col, higher, label = METRICS[metric]
    print(f"\n### {label}")
    display(t.round(3).sort_values(t.columns[0], ascending=not higher))

## A2. Per-platform rankings

In [ ]:
for metric in ("accuracy", "latency"):
    t = pivot(metric)
    if t is None or t.empty:
        continue
    col, higher, label = METRICS[metric]
    n = len(t.columns)
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, max(3.6, 0.32 * len(t))), squeeze=False)
    for ax, p in zip(axes[0], t.columns):
        d = t[p].dropna().sort_values(ascending=not higher)
        ax.barh(np.arange(len(d)), d.values,
                color=[PALETTE.get(m, "grey") for m in d.index], height=0.66)
        ax.set_yticks(np.arange(len(d))); ax.set_yticklabels(d.index, fontsize=7)
        ax.invert_yaxis(); ax.set_xlabel(label); ax.set_title(p, fontsize=10)
    fig.suptitle(f"{label} by platform", y=1.02)
    fig.tight_layout(); save(fig, f"A2_{metric}_by_platform"); plt.show()

## A3. Accuracy against latency — the deployment trade-off

The Pareto front is the set of models for which no other model is both faster and more accurate.
A front that differs between platforms is direct evidence against ranking transfer, and is the
single most useful figure for a deployment decision.

In [ ]:
acc_col, _, _ = METRICS["accuracy"]
lat_col, _, _ = METRICS["latency"]

if {acc_col, lat_col}.issubset(results.columns):
    ps = [p for p in PLATFORM_ORDER if results[results.platform == p][lat_col].notna().any()]
    if ps:
        n = len(ps)
        fig, axes = plt.subplots(1, n, figsize=(6.4 * n, 5.4), squeeze=False)
        fronts = {}
        for ax, p in zip(axes[0], ps):
            d = results[results.platform == p].dropna(subset=[acc_col, lat_col])
            for _, r in d.iterrows():
                ax.scatter(r[lat_col], r[acc_col], s=72, color=PALETTE.get(r.model, "grey"),
                           edgecolor="k", lw=0.5, zorder=3)
                ax.annotate(r.model, (r[lat_col], r[acc_col]), fontsize=6,
                            xytext=(4, 3), textcoords="offset points")
            f = pareto_front(d, lat_col, acc_col)
            fronts[p] = list(f.model)
            ax.plot(f[lat_col], f[acc_col], "k--", lw=1.2, alpha=0.7, zorder=2,
                    label=f"Pareto front ({len(f)})")
            ax.set(xlabel="median latency (ms)", ylabel="top-1 (%)",
                   title=f"{p}  [{PLATFORMS[p]['precision']}]")
            ax.legend(loc="lower right")
        fig.suptitle("Accuracy against latency — Pareto frontier per platform", y=1.02)
        fig.tight_layout(); save(fig, "A3_pareto_accuracy_latency"); plt.show()

        print("Pareto-optimal models per platform:")
        for p, models in fronts.items():
            print(f"  {p:16s} {models}")
        if len(fronts) > 1:
            common = set.intersection(*(set(v) for v in fronts.values()))
            union = set.union(*(set(v) for v in fronts.values()))
            print(f"\n  on every front : {sorted(common)}")
            print(f"  on some front  : {sorted(union - common)}  <- platform-specific optima")

## A4. Do operation counts predict latency?

The standard proxy for efficiency in the architecture literature is the multiply-accumulate
count. This tests it directly, per platform. A weak correlation is itself a reportable result,
and the architectures that deviate most are the interesting cases.

In [ ]:
if "GMACs" in results.columns and lat_col in results.columns:
    ps = [p for p in PLATFORM_ORDER if results[results.platform == p][lat_col].notna().any()]
    n = len(ps)
    fig, axes = plt.subplots(1, n, figsize=(5.6 * n, 5), squeeze=False)
    corr_rows = []
    for ax, p in zip(axes[0], ps):
        d = results[results.platform == p].dropna(subset=["GMACs", lat_col])
        if len(d) < 3:
            continue
        rho = d.GMACs.corr(d[lat_col], method="spearman")
        tau = d.GMACs.corr(d[lat_col], method="kendall")
        r2 = d.params_M.corr(d[lat_col], method="spearman") if "params_M" in d else np.nan
        corr_rows.append({"platform": p, "spearman_macs": round(rho, 3),
                          "kendall_macs": round(tau, 3), "spearman_params": round(r2, 3),
                          "n": len(d)})
        for _, r in d.iterrows():
            ax.scatter(r.GMACs, r[lat_col], s=66, color=PALETTE.get(r.model, "grey"),
                       edgecolor="k", lw=0.5)
            ax.annotate(r.model, (r.GMACs, r[lat_col]), fontsize=5.8,
                        xytext=(3, 3), textcoords="offset points")
        ax.set(xscale="log", yscale="log", xlabel="multiply-accumulates (G)",
               ylabel="median latency (ms)", title=f"{p}\nSpearman {rho:+.2f}")
    fig.suptitle("Theoretical cost against measured latency", y=1.03)
    fig.tight_layout(); save(fig, "A4_macs_vs_latency"); plt.show()
    display(pd.DataFrame(corr_rows))
    print("A correlation well below 1 means operation counts are a poor efficiency proxy on "
          "that platform. Depthwise-separable and channel-shuffle designs are the usual cause: "
          "they are memory-bound, and memory traffic is not counted in MACs.")

---
# Part B — Does ranking transfer?

The central question. Rankings are compared with **Kendall's tau-b**, which counts concordant
and discordant pairs and handles ties. Interpretation:

| tau | meaning |
|---|---|
| 1.0 | identical ordering |
| ~0.8 | broadly the same, a few local swaps |
| ~0.5 | substantial reordering |
| 0.0 | unrelated orderings |

Tau is computed **per metric**: latency rankings may transfer while accuracy rankings do not,
and conflating them would hide that.

## B1. The noise floor

A cross-platform tau below 1 is only meaningful relative to how reproducible a ranking is on a
**single** platform. Repeat one platform's measurement into a second CSV and register it as
`"<platform> (repeat)"`, then the cell below reports the ceiling that cross-platform values
should be judged against.

In [ ]:
repeats = [(a, b) for a in PLATFORM_ORDER for b in PLATFORM_ORDER
           if a != b and b.startswith(a.split(" (")[0]) and "repeat" in b.lower()]
if repeats:
    print("Repeat-measurement agreement (the ceiling for cross-platform tau):")
    for metric in METRICS:
        r = ranked(metric)
        if r is None:
            continue
        for a, b in repeats:
            if {a, b}.issubset(r.columns):
                print(f"  {metric:11s} {a} vs {b}: tau = "
                      f"{r[a].corr(r[b], method='kendall'):+.3f}")
else:
    print("No repeat measurement registered.")
    print("Without it there is no noise floor, and a tau of, say, 0.85 cannot be distinguished")
    print("from measurement variation. Re-run one platform and register it as a repeat before")
    print("drawing conclusions about imperfect transfer.")

## B2. Rank-correlation matrices

In [ ]:
TAU_CMAP = LinearSegmentedColormap.from_list("tau", ["#b2182b", "#f7f7f7", "#2166ac"])

available = {m: ranked(m) for m in METRICS}
available = {m: r for m, r in available.items()
             if r is not None and r.shape[1] >= 2 and len(r) >= 3}

if not available:
    print("At least two platforms with a shared set of models are needed. "
          "Register more platforms in PLATFORMS as results arrive.")
else:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(5.4 * n, 4.8), squeeze=False)
    tau_tables = {}
    for ax, (metric, r) in zip(axes[0], available.items()):
        tau = r.corr(method="kendall")
        tau_tables[metric] = tau
        heat(ax, tau, TAU_CMAP, vmin=-1, vmax=1, fmt="{:+.2f}", cbar_label="Kendall tau")
        ax.set_title(f"{metric}  (n={len(r)} models)")
    fig.suptitle("Do rankings transfer? Pairwise Kendall tau between platforms", y=1.04)
    fig.tight_layout(); save(fig, "B2_tau_matrices"); plt.show()

    print("Off-diagonal tau, summarised:")
    for metric, tau in tau_tables.items():
        vals = tau.values[np.triu_indices_from(tau.values, k=1)]
        if vals.size:
            print(f"  {metric:11s} min {vals.min():+.3f}  mean {vals.mean():+.3f}  "
                  f"max {vals.max():+.3f}")

### B2.1 Bootstrap confidence intervals

A tau computed over roughly sixteen models carries real sampling uncertainty. Resampling models
with replacement gives an interval, so "rankings differ" is separated from "too few models to
tell".

In [ ]:
def tau_ci(ranks_a, ranks_b, n_boot=2000, seed=0):
    """Bootstrap a confidence interval for Kendall tau between two rankings.

    Models are resampled with replacement, which treats the model set as a sample from the
    population of architectures one might have chosen.

    Args:
        ranks_a: Ranks on the first platform.
        ranks_b: Ranks on the second platform, same index.
        n_boot: Resamples.
        seed: Seed for reproducibility.

    Returns:
        Point estimate, 2.5th percentile and 97.5th percentile.
    """
    rng = np.random.default_rng(seed)
    a, b = np.asarray(ranks_a, float), np.asarray(ranks_b, float)
    point = pd.Series(a).corr(pd.Series(b), method="kendall")
    draws = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(a), len(a))
        if len(np.unique(a[idx])) < 2:        # a degenerate resample has undefined tau
            continue
        draws.append(pd.Series(a[idx]).corr(pd.Series(b[idx]), method="kendall"))
    if not draws:
        return point, np.nan, np.nan
    return point, float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


if available:
    rows = []
    for metric, r in available.items():
        cols = list(r.columns)
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                t, lo, hi = tau_ci(r[cols[i]], r[cols[j]])
                rows.append({"metric": metric, "A": cols[i], "B": cols[j],
                             "tau": round(t, 3), "ci_lo": round(lo, 3),
                             "ci_hi": round(hi, 3), "n_models": len(r),
                             # An interval excluding 1.0 is evidence the orderings genuinely
                             # differ rather than coinciding within sampling noise.
                             "differs_from_perfect": bool(hi < 1.0)})
    ci = pd.DataFrame(rows)
    display(ci)
    ci.to_csv(BENCH / "rank_transfer_tau.csv", index=False)
    print(f"wrote {BENCH / 'rank_transfer_tau.csv'}")

## B3. Where does the ranking move?

The tau value says *how much* the ordering changes; this says *which models* move. A line that
crosses many others is an architecture whose standing is platform-dependent, which is exactly
what a deployment decision needs to know.

In [ ]:
for metric in ("latency", "accuracy"):
    r = ranked(metric)
    if r is None or r.shape[1] < 2:
        continue
    fig, ax = plt.subplots(figsize=(1.9 * r.shape[1] + 4.5, max(4.5, 0.32 * len(r))))
    x = np.arange(r.shape[1])
    for m in r.index:
        ax.plot(x, r.loc[m].values, "-o", ms=5, lw=1.6,
                color=PALETTE.get(m, "grey"), alpha=0.9)
        ax.annotate(m, (x[0], r.loc[m].values[0]), fontsize=7,
                    xytext=(-7, 0), textcoords="offset points", ha="right", va="center")
        ax.annotate(m, (x[-1], r.loc[m].values[-1]), fontsize=7,
                    xytext=(7, 0), textcoords="offset points", ha="left", va="center")
    ax.set_xticks(x); ax.set_xticklabels(r.columns, rotation=20, ha="right")
    ax.set_ylabel("rank (1 = best)"); ax.invert_yaxis()
    ax.set_title(f"Rank movement across platforms — {metric}")
    ax.set_xlim(-1.1, x[-1] + 1.1); ax.margins(x=0.25)
    fig.tight_layout(); save(fig, f"B3_rank_movement_{metric}"); plt.show()

    if r.shape[1] >= 2:
        shift = pd.DataFrame({
            "best_rank": r.min(axis=1).astype(int),
            "worst_rank": r.max(axis=1).astype(int),
            "rank_range": (r.max(axis=1) - r.min(axis=1)).astype(int),
            "rank_std": r.std(axis=1).round(2),
        }).sort_values("rank_range", ascending=False)
        print(f"\nMost platform-sensitive on {metric}:")
        display(shift.head(10))

## B4. Relative performance heatmap

Each column is normalised to its own best model, which removes the large absolute differences
between platforms and exposes the *relative* standing of each architecture. A row of uniform
colour is a model that behaves consistently; a row that varies is one that does not.

In [ ]:
for metric in ("latency", "throughput", "accuracy"):
    t = pivot(metric)
    if t is None or t.shape[1] < 1 or t.dropna(how="all").empty:
        continue
    col, higher, label = METRICS[metric]
    t = t.dropna(how="all")
    # Normalise so 1.0 is always the best value in that column, whichever direction is better.
    norm = t.div(t.max(axis=0), axis=1) if higher else t.div(t.min(axis=0), axis=1)
    order = norm.mean(axis=1).sort_values(ascending=not higher).index
    norm = norm.loc[order]
    fig, ax = plt.subplots(figsize=(1.6 * norm.shape[1] + 4, max(4, 0.34 * len(norm))))
    heat(ax, norm, "viridis" if higher else "viridis_r",
         fmt="{:.2f}", cbar_label=f"relative {metric} (1.0 = best in column)")
    ax.set_title(f"Relative {label} — normalised within each platform")
    fig.tight_layout(); save(fig, f"B4_relative_{metric}"); plt.show()

## B5. Quantisation cost

Accuracy lost relative to the FP32 reference. This is a property of the architecture interacting
with each vendor's quantisation algorithm, and it is a distinct question from speed. The
calibration *data* is identical across backends by construction; the calibration *algorithm* is
not, and that difference is part of what this measures.

In [ ]:
REFERENCE = next((p for p in PLATFORM_ORDER if PLATFORMS[p]["precision"] == "fp32"), None)

if REFERENCE and acc_col in results.columns:
    t = pivot("accuracy")
    others = [p for p in t.columns if p != REFERENCE]
    if others:
        delta = t[others].sub(t[REFERENCE], axis=0).dropna(how="all")
        fig, ax = plt.subplots(figsize=(1.7 * len(others) + 4, max(4, 0.34 * len(delta))))
        lim = float(np.nanmax(np.abs(delta.values))) or 1.0
        heat(ax, delta.sort_values(others[0]),
             LinearSegmentedColormap.from_list("d", ["#b2182b", "#f7f7f7", "#2166ac"]),
             vmin=-lim, vmax=lim, fmt="{:+.1f}",
             cbar_label=f"top-1 change vs {REFERENCE} (pp)")
        ax.set_title(f"Accuracy change relative to {REFERENCE}")
        fig.tight_layout(); save(fig, "B5_quantisation_cost"); plt.show()
        display(delta.round(2))
        print("Large losses concentrated in particular lineages indicate architectures that "
              "quantise badly — a different and subtler failure than one that does not compile.")
else:
    print(f"No FP32 reference platform registered; quantisation cost needs one.")

---
# Part C — Deployability

Whether an architecture runs at all, and how much of it the accelerator actually executes. For
fixed-function hardware this is often more decisive than speed: a model that compiles but falls
back to the host CPU is not accelerated in any useful sense.

Populated from the compile logs once the hardware campaign begins.

In [ ]:
compile_rows = []
for label, spec in PLATFORMS.items():
    for name in (f"compile_log_{label.split()[0].lower()}.csv", "compile_log.csv"):
        p = BENCH.parent.parent / "binaries" / label.split()[0].lower() / TASK / name
        if p.exists():
            d = pd.read_csv(p).assign(platform=label)
            compile_rows.append(d)
            break

if compile_rows:
    comp = pd.concat(compile_rows, ignore_index=True)
    if {"model", "platform", "status"}.issubset(comp.columns):
        status = comp.pivot_table(index="model", columns="platform", values="status",
                                  aggfunc="first")
        display(status)
    # tpu_frac is the share of graph operations the Edge TPU compiler accepted; the remainder
    # runs on the host CPU.
    if "tpu_frac" in comp.columns:
        f = comp.pivot_table(index="model", columns="platform", values="tpu_frac",
                             aggfunc="first").dropna(how="all")
        if not f.empty:
            fig, ax = plt.subplots(figsize=(1.7 * f.shape[1] + 4, max(4, 0.34 * len(f))))
            heat(ax, f.sort_values(f.columns[0]), "RdYlGn", vmin=0, vmax=1, fmt="{:.0%}",
                 cbar_label="fraction of operations on the accelerator")
            ax.set_title("Accelerator coverage — the rest falls back to the host CPU")
            fig.tight_layout(); save(fig, "C1_accelerator_coverage"); plt.show()
else:
    print("No compile logs yet. Once the hardware campaign runs, this section reports:")
    print("  - compile success or failure per model and platform, with the offending operation")
    print("  - the fraction of operations mapped to the accelerator versus the host CPU")
    print("  - multi-context models on Hailo, which pay a weight-swapping penalty")
    print("\nA model that fails to compile is a RESULT, not missing data. Report the funnel: "
          "how many architectures survived each stage, per platform.")

### C1. Graph complexity as a predictor of deployability

The ONNX operation histograms written by the export stage are available before any hardware is
touched. Data-movement operations — transpose, reshape, gather, slice — are the usual reason a
graph is rejected or partially offloaded, so counting them early gives a testable prediction.

In [ ]:
onnx_dir = ROOT / "checkpoint" / "onnx" / TASK
sidecars = sorted(onnx_dir.glob("*.json")) if onnx_dir.is_dir() else []

if sidecars:
    MOVEMENT = {"Transpose", "Reshape", "Gather", "Slice", "Concat", "Split", "Shape"}
    rows = []
    for p in sidecars:
        meta = json.loads(p.read_text())
        ops = meta.get("op_types", {}) or {}
        total = sum(ops.values()) or 1
        rows.append({"model": meta.get("model", p.stem),
                     "n_nodes": meta.get("n_nodes"),
                     "n_op_types": meta.get("n_op_types"),
                     "movement_ops": sum(v for k, v in ops.items() if k in MOVEMENT),
                     "movement_frac": round(
                         sum(v for k, v in ops.items() if k in MOVEMENT) / total, 3),
                     "opset": meta.get("opset")})
    graph = pd.DataFrame(rows).sort_values("movement_frac", ascending=False)
    display(graph)

    fig, ax = plt.subplots(figsize=(9, max(3.6, 0.34 * len(graph))))
    y = np.arange(len(graph))
    ax.barh(y, graph.movement_frac, color=[PALETTE.get(m, "grey") for m in graph.model],
            height=0.66)
    ax.set_yticks(y); ax.set_yticklabels(graph.model, fontsize=7.5); ax.invert_yaxis()
    ax.set_xlabel("share of graph nodes that are data-movement operations")
    ax.set_title("Predicted difficulty for fixed-function accelerators")
    fig.tight_layout(); save(fig, "C2_graph_movement_ops"); plt.show()
    print("Prediction to test against the compile logs: the models at the top of this chart "
          "are the ones most likely to fail or fall back to the host.")
else:
    print(f"No ONNX sidecars under {onnx_dir}; run the export stage first.")

## D. Export the combined results table

In [ ]:
keep = [c for c in ["model", "platform", "precision", "kind", "res", "top1", "top5",
                    "lat_p50_ms", "lat_p95_ms", "lat_p99_ms", "throughput_img_s",
                    "params_M", "GMACs", "peak_memory_bs1_mb", "recipe_digest"]
        if c in results.columns]
wide = results[keep].sort_values(["platform", "top1"], ascending=[True, False])
out = BENCH / "cross_platform_results.csv"
wide.to_csv(out, index=False)
print(f"wrote {out}  ({len(wide)} rows)")
display(wide.head(30))

## Notes for the write-up

**On the central question.** Report tau per metric, with confidence intervals, and always
against the single-platform noise floor from B1. "Rankings do not transfer" is only defensible
once cross-platform tau is shown to sit below the repeat-measurement ceiling.

**On precision.** The `precision` column is not decorative. TensorRT INT8 is mixed precision
chosen by the builder, while the Edge TPU and Hailo are integer-only. Any table comparing them
must say so, or a reviewer will reasonably object that two different things are being compared.

**On the reference platform.** A development GPU is not an edge device. It belongs in this
analysis as an unconstrained upper bound and as a fifth ranking for developing the method, not
in a like-for-like table of edge hardware.

**On failures.** A model that does not compile is a deployability result. Report the funnel —
how many architectures survived each stage on each platform — rather than silently analysing
only the survivors, which would bias the sample towards simple architectures.